# 02 - Preprocessing, Chronological Split, and Sequence Windowing

Turns the raw table into the two representations used downstream:

1. A **flat feature table** (13 predictors: 8 static meteorological + Hr + 4 lag
   temperatures) matching the internship's own final feature set, for the
   classical-ML baselines refit on this project's split (notebook 03).
2. A **(4-timestep x 10-channel) sequence tensor** per row, for the CNN/LSTM/
   CNN-LSTM models (notebook 04).

Both share the same chronological, burst-aware split and the same train-only-fit
scalers. See `reports/phase2_design.md` for the full rationale.


In [1]:
import numpy as np
import pandas as pd

from src.data import (
    load_raw, detect_bursts, block_chronological_split,
    LAG_COLS, STATIC_FEATURE_COLS, FLAT_FEATURE_COLS, TARGET_COLS,
)
from src.windowing import SequenceScalers, build_sequence_tensor, N_TIMESTEPS, N_CHANNELS

pd.set_option("display.width", 120)

df = load_raw()
print("Raw rows:", len(df))

Raw rows: 300


## Drop the one row with an incomplete lag block

One row (2020-08-01) has a missing `T(i-1)` — it is the first day of its own burst
in this curated sample, so no valid antecedent temperature exists for it in the
source data. Dropping it (300 -> 299 rows) rather than imputing avoids fabricating
a physical temperature value; losing 1/300 rows is negligible.


In [2]:
missing_lag = df[df["T(i-1)"].isnull()]
print("Rows with missing T(i-1):")
print(missing_lag[["Date", "Hr", "dbt", "wbt"]])

df = df.dropna(subset=["T(i-1)"]).reset_index(drop=True)
print()
print("Rows after drop:", len(df))

Rows with missing T(i-1):
          Date  Hr   dbt   wbt
290 2020-08-01   9  35.8  29.6

Rows after drop: 299


In [3]:
df = detect_bursts(df)
print("Bursts after drop:", df["burst_id"].nunique())

Bursts after drop: 71


## Chronological, burst-aware split


In [4]:
train_df, val_df, test_df = block_chronological_split(df, train_frac=0.70, val_frac=0.15)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {len(part):3d} rows ({len(part)/len(df):5.1%}) | "
          f"{part.Date.min().date()} -> {part.Date.max().date()} | "
          f"{part.burst_id.nunique()} bursts")

train: 210 rows (70.2%) | 1980-04-20 -> 2005-06-19 | 49 bursts
val  :  46 rows (15.4%) | 2005-08-28 -> 2014-08-22 | 11 bursts
test :  43 rows (14.4%) | 2015-05-21 -> 2020-08-31 | 11 bursts


## Representation 1 — flat feature table (for classical-ML baselines)

Matches the internship's own final predictor set exactly (`FLAT_FEATURE_COLS`),
so the baseline rematch in notebook 03 isolates the effect of the *split strategy*
and re-tuning, not a different feature set.


In [5]:
X_train_flat = train_df[FLAT_FEATURE_COLS].copy()
X_val_flat = val_df[FLAT_FEATURE_COLS].copy()
X_test_flat = test_df[FLAT_FEATURE_COLS].copy()

y_train = train_df[TARGET_COLS].copy()
y_val = val_df[TARGET_COLS].copy()
y_test = test_df[TARGET_COLS].copy()

print("Flat feature matrix shapes:", X_train_flat.shape, X_val_flat.shape, X_test_flat.shape)
print()
print("Feature columns (13):", list(X_train_flat.columns))
X_train_flat.describe().T[["mean", "std", "min", "max"]]

Flat feature matrix shapes: (210, 13) (46, 13) (43, 13)

Feature columns (13): ['specific_humidity_500hPa', 'geopotential_height_500hPa', 'surface_sensible_heat_flux', 'surface_latent_heat_flux', 'Wind_500hPa_ms', 'OLR_Wm2', 'TCC', 'Urban_Footprint', 'Hr', 'T(i-4)', 'T(i-3)', 'T(i-2)', 'T(i-1)']


## Representation 2 — sequence tensor (for CNN / LSTM / CNN-LSTM)

Scalers are fit on **train only**, then applied unchanged to val/test — the same
discipline the internship used for its `StandardScaler`, extended to the sequence
representation.


In [6]:
scalers = SequenceScalers().fit(train_df)

X_train_seq, y_train_seq = build_sequence_tensor(train_df, scalers)
X_val_seq, y_val_seq = build_sequence_tensor(val_df, scalers)
X_test_seq, y_test_seq = build_sequence_tensor(test_df, scalers)

print("Sequence tensor shapes (n, timesteps, channels):")
print(" train:", X_train_seq.shape, "y:", y_train_seq.shape)
print(" val:  ", X_val_seq.shape, "y:", y_val_seq.shape)
print(" test: ", X_test_seq.shape, "y:", y_test_seq.shape)
print()
print("Expected (timesteps, channels):", (N_TIMESTEPS, N_CHANNELS))
print("NaNs in any split:", any(np.isnan(a).any() for a in [X_train_seq, X_val_seq, X_test_seq]))

Sequence tensor shapes (n, timesteps, channels):
 train: (210, 4, 10) y: (210, 2)
 val:   (46, 4, 10) y: (46, 2)
 test:  (43, 4, 10) y: (43, 2)

Expected (timesteps, channels): (4, 10)
NaNs in any split: False


In [7]:
# Scaling sanity check: train targets should be ~N(0,1); val/test need not be,
# since the scaler is fit on train only (that's the point).
print("Train target scaled mean/std (expect ~0 / ~1):", y_train_seq.mean(axis=0), y_train_seq.std(axis=0))
print("Val   target scaled mean/std:", y_val_seq.mean(axis=0), y_val_seq.std(axis=0))
print("Test  target scaled mean/std:", y_test_seq.mean(axis=0), y_test_seq.std(axis=0))

Train target scaled mean/std (expect ~0 / ~1): [0.000000e+00 9.082613e-09] [1. 1.]
Val   target scaled mean/std: [-0.27708125  0.10191422] [1.1383293 1.0640593]
Test  target scaled mean/std: [0.15879242 0.21624829] [1.0857605 0.6868139]


## Leakage re-check on the final split arrays

Belt-and-braces: re-verify no burst_id appears in more than one split, and that
splits are strictly chronologically ordered, on the *post-dropna* dataframes
actually used to build the arrays above (not just the pre-dropna check in
notebook 01).


In [8]:
train_ids = set(train_df.burst_id)
val_ids = set(val_df.burst_id)
test_ids = set(test_df.burst_id)

assert not (train_ids & val_ids), "train/val burst overlap!"
assert not (val_ids & test_ids), "val/test burst overlap!"
assert not (train_ids & test_ids), "train/test burst overlap!"
assert train_df.Date.max() < val_df.Date.min(), "train/val not chronologically ordered!"
assert val_df.Date.max() < test_df.Date.min(), "val/test not chronologically ordered!"
print("No burst overlap across splits. Splits are strictly chronologically ordered.")

No burst overlap across splits. Splits are strictly chronologically ordered.


## Persist processed arrays

Saved under `data/processed/` (git-ignored — regenerate by re-running this
notebook; nothing here is a source of truth that isn't reproducible from
`data/raw/` + this notebook).


In [9]:
import pickle
from pathlib import Path

out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

np.savez(
    out_dir / "sequence_arrays.npz",
    X_train=X_train_seq, y_train=y_train_seq,
    X_val=X_val_seq, y_val=y_val_seq,
    X_test=X_test_seq, y_test=y_test_seq,
)

X_train_flat.to_csv(out_dir / "X_train_flat.csv", index=False)
X_val_flat.to_csv(out_dir / "X_val_flat.csv", index=False)
X_test_flat.to_csv(out_dir / "X_test_flat.csv", index=False)
y_train.to_csv(out_dir / "y_train.csv", index=False)
y_val.to_csv(out_dir / "y_val.csv", index=False)
y_test.to_csv(out_dir / "y_test.csv", index=False)

with open(out_dir / "scalers.pkl", "wb") as f:
    pickle.dump(scalers, f)

print("Saved sequence_arrays.npz, flat CSVs, and scalers.pkl to", out_dir)

Saved sequence_arrays.npz, flat CSVs, and scalers.pkl to data\processed
